# E2 · Parameter learning with missing data

Compare complete-case MLE, mode imputation followed by MLE, and EM on DAG. All methods use the same encoded splits, state order, and add-one smoothing. Each rate and seed generates one training mask shared by all three methods.

We add 0%, 20%, or 40% missingness independently among observed cells in the two condition-related columns. The 0% setting retains the original missing values. The three seeds repeat the mask draw on the same split, so their variability describes sensitivity to masking.

In [15]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd

from parameter_learning import fit_em, fit_mle
from parameter_learning.e2_handoff import MASK_COLUMNS, mcar_mask
from structure.bn import dag_from_edges

In [20]:
root =  Path('c:/Users/jessi/OneDrive/Documents/GitHub/jessie/CSE250a-BayesCar/')
path = root / "structure" / "outputs" / "dags.json"
artifact = json.loads(path.read_text(encoding="utf-8"))
card = artifact["cardinality"]
dag = dag_from_edges(card, artifact["edges"]["expert"])
prepared = root / "dataset/prepared"
schema = json.loads((prepared / "schema.json").read_text(encoding="utf-8"))["columns"]
def load_split(name):
    return pd.read_csv(prepared / f"{name}.csv", index_col="source_row", dtype="Int64")
train = load_split("train")
test = load_split("test")
print(f"Training rows: {len(train):,} | Test rows: {len(test):,}")
display(train[MASK_COLUMNS].isna().mean().rename("original_train_missing_fraction"))


Training rows: 80,000 | Test rows: 10,000


service_history_quality    0.039512
overall_condition_score    0.009862
Name: original_train_missing_fraction, dtype: float64

## Fit and score one shared mask

Complete-case Maximum Likelihood Estimation (MLE) removes training rows containing any missing model variables. Pattern + MLE fills in missing training values ​​using the most common observed states in each column, breaking ties based on state codes. The EM algorithm starts with the CPT of Pattern + MLE and directly fits the masked observations. We record its stopping state and number of iterations.

Test Log-Likelihood uses the same raw test evidence for each model and marginalizes missing values. To improve recovery accuracy, we create a separate fixed 20% test mask. Manually hidden columns remain masked when querying the posterior distribution. Recovery scores only compute intentionally hidden test cells with known raw values.

Parameter Diagnostics computes the average KL divergence between nodes and their parent configurations and the unmasked full-case reference CPT. The reference value is an estimate based on the observed data, and the raw missing values ​​are inaccessible. A smaller KL value indicates higher agreement with the reference value.

In [22]:
ALPHA = 1.0
RATES = (0.0, 0.2, 0.4)
SEEDS = (0, 1, 2)
METHODS = ("Complete-case MLE", "Mode + MLE", "EM")
reference = fit_mle(dag, train, card, alpha=ALPHA, missing="complete_case")
masked_test, hidden_test = mcar_mask(test, MASK_COLUMNS, rate=0.2, seed=2026)


In [23]:
def mean_cpt_kl(model):
    node_kl = []
    for node, p in reference.cpts.items():
        q = model.cpts[node]
        node_kl.append(np.sum(p * (np.log(p) - np.log(q)), axis=1).mean())
    return float(np.mean(node_kl))

In [24]:
def run_comparison(rate, seed):
    masked_train, hidden_train = mcar_mask(train, MASK_COLUMNS, rate, seed)
    assert masked_train.isna().equals(train.isna() | hidden_train)
    assert not (hidden_train & train.isna()).any().any()

    modes = masked_train.mode(dropna=True).iloc[0]
    if modes.isna().any():
        raise ValueError("Mode imputation requires an observed value in every column.")

    complete_case = fit_mle(dag, masked_train, card, alpha=ALPHA, missing="complete_case")
    mode_mle = fit_mle(dag, masked_train.fillna(modes), card, alpha=ALPHA)
    em = fit_em(dag, masked_train, card, alpha=ALPHA, init_cpts=mode_mle.cpts,
        seed=seed, max_iter=500, tol=1e-7,)
    models = (complete_case, mode_mle, em.model)
    counts = (len(masked_train.dropna()), len(masked_train), len(masked_train))
    scores = []
    for method, model, used in zip(METHODS, models, counts):
        row = {
            "mask_rate": rate, "mask_seed": seed, "method": method,"train_rows_used": used, 
            "test_ll_per_row": model.log_likelihood(test).mean(), "mean_cpt_kl": mean_cpt_kl(model),
        }
        for col in MASK_COLUMNS:
            hidden = hidden_test[col]
            evidence = masked_test.loc[hidden]
            truth = test.loc[hidden, col].to_numpy(dtype=int)
            predicted = model.posterior(evidence, col).argmax(axis=1)
            row[f"{col}_accuracy"] = np.mean(predicted == truth)
        scores.append(row)
    diagnostic = {
        "mask_rate": rate, "mask_seed": seed,
        "em_converged": em.converged, "em_iterations": em.n_iter,
    }
    for col in MASK_COLUMNS:
        diagnostic[f"{col}_added_fraction"] = hidden_train[col].sum() / train[col].count()
    return scores, diagnostic

## Compare results across masking seeds

Higher test LL and recovery accuracy are better. Lower CPT KL means closer agreement with the reference. At 0% extra masking, the three seeds use identical training evidence and the same EM initialization. For 20% and 40%, the paired differences count how often EM improves on complete-case MLE under the same mask.

In [25]:
runs = []
diagnostics = []
for rate in RATES:
    for seed in SEEDS:
        scores, diagnostic = run_comparison(rate, seed)
        runs.extend(scores)
        diagnostics.append(diagnostic)
        print(f"Rate={rate:.0%}, seed={seed}: {diagnostic['em_iterations']} EM iterations")


Rate=0%, seed=0: 3 EM iterations
Rate=0%, seed=1: 3 EM iterations
Rate=0%, seed=2: 3 EM iterations
Rate=20%, seed=0: 7 EM iterations
Rate=20%, seed=1: 7 EM iterations
Rate=20%, seed=2: 8 EM iterations
Rate=40%, seed=0: 14 EM iterations
Rate=40%, seed=1: 13 EM iterations
Rate=40%, seed=2: 14 EM iterations


In [26]:
results = pd.DataFrame(runs)
diagnostics = pd.DataFrame(diagnostics)
metrics = ["train_rows_used", "test_ll_per_row", "mean_cpt_kl",
*(f"{col}_accuracy" for col in MASK_COLUMNS),]
summary = results.groupby(["mask_rate", "method"], sort=False)[metrics].agg(["mean", "std"])
display(summary.round(5))
display(diagnostics.round(4))


train_rows_used           test_ll_per_row  \
                                       mean       std            mean   
mask_rate method                                                        
0.0       Complete-case MLE     76081.00000    0.0000       -10.86407   
          Mode + MLE            80000.00000    0.0000       -10.86682   
          EM                    80000.00000    0.0000       -10.86374   
0.2       Complete-case MLE     48699.33333   76.6442       -10.86595   
          Mode + MLE            80000.00000    0.0000       -10.99005   
          EM                    80000.00000    0.0000       -10.86431   
0.4       Complete-case MLE     27441.33333  103.5825       -10.87056   
          Mode + MLE            80000.00000    0.0000       -11.22263   
          EM                    80000.00000    0.0000       -10.86666   

                                     mean_cpt_kl           \
                                 std        mean      std   
mask_rate method                                            
0.0       Complete-case MLE  0.00000     0.00000  0.00000   
          Mode + MLE         0.00000     0.00178  0.00000   
          EM                 0.00000     0.00004  0.00000   
0.2       Complete-case MLE  0.00027     0.00089  0.00016   
          Mode + MLE         0.02434     0.01427  0.00461   
          EM                 0.00018     0.00058  0.00017   
0.4       Complete-case MLE  0.00043     0.00299  0.00024   
          Mode + MLE         0.04262     0.03115  0.00560   
          EM                 0.00008     0.00195  0.00016   

                            service_history_quality_accuracy           \
                                                        mean      std   
mask_rate method                                                        
0.0       Complete-case MLE                          0.43989  0.00000   
          Mode + MLE                                 0.43881  0.00000   
          EM                                         0.43989  0.00000   
0.2       Complete-case MLE                          0.43971  0.00031   
          Mode + MLE                                 0.43881  0.00000   
          EM                                         0.43989  0.00054   
0.4       Complete-case MLE                          0.43917  0.00359   
          Mode + MLE                                 0.43881  0.00000   
          EM                                         0.43917  0.00243   

                            overall_condition_score_accuracy           
                                                        mean      std  
mask_rate method                                                       
0.0       Complete-case MLE                          0.74057  0.00000  
          Mode + MLE                                 0.74057  0.00000  
          EM                                         0.73955  0.00000  
0.2       Complete-case MLE                          0.74023  0.00106  
          Mode + MLE                                 0.73530  0.00324  
          EM                                         0.74040  0.00156  
0.4       Complete-case MLE                          0.74023  0.00156  
          Mode + MLE                                 0.59837  0.02036  
          EM                                         0.74125  0.00212

,mask_rate,mask_seed,em_converged,em_iterations,service_history_quality_added_fraction,overall_condition_score_added_fraction
0,0.0,0,True,3,0.0000,0.0000
1,0.0,1,True,3,0.0000,0.0000
2,0.0,2,True,3,0.0000,0.0000
3,0.2,0,True,7,0.2007,0.1998
4,0.2,1,True,7,0.1986,0.2002
5,0.2,2,True,8,0.2010,0.1992
6,0.4,0,True,14,0.4003,0.3998
7,0.4,1,True,13,0.4019,0.3998
8,0.4,2,True,14,0.4009,0.3987


In [28]:
paired = results.pivot(index=["mask_rate", "mask_seed"], columns="method",
                       values=["test_ll_per_row", "mean_cpt_kl"],)
paired_wins = pd.DataFrame({
    "em_higher_test_ll": paired["test_ll_per_row"]["EM"] > paired["test_ll_per_row"]["Complete-case MLE"],
    "em_lower_cpt_kl": paired["mean_cpt_kl"]["EM"] < paired["mean_cpt_kl"]["Complete-case MLE"],
})
display(paired_wins.groupby(level="mask_rate").sum())

,em_higher_test_ll,em_lower_cpt_kl
mask_rate,,
0.0,3,0
0.2,3,3
0.4,3,3


## E2 conclusion

We ran E2 to measure how missing service history and condition scores affect parameter learning in our used-car Bayesian network. The learned conditional probability tables (CPTs) support downstream inference, including queries about listing-price tiers. We held the expert DAG, training masks, and test evidence fixed across methods so the comparison isolates how each estimator handles missing values.

The masking probabilities apply independently to observed cells in two training columns. At 0% additional masking, the original missing values remain. Test log-likelihood is averaged across three masking seeds and reported in nats per row. Higher is better.

| Added masking | Complete-case MLE | Mode imputation + MLE | EM | Complete-case training rows retained |
| --- | ---: | ---: | ---: | ---: |
| 0% | -10.86407 | -10.86682 | **-10.86374** | 76,081 / 80,000 (95.1%) |
| 20% | -10.86595 | -10.99005 | **-10.86431** | 48,699 / 80,000 (60.9%) |
| 40% | -10.87056 | -11.22263 | **-10.86666** | 27,441 / 80,000 (34.3%) |

- **0% added masking.** Complete-case MLE is a simple baseline and retains most rows. EM gains only 0.00033 nats per row over it and converges in three iterations. Mode imputation keeps all rows, with a slightly lower test likelihood.
- **20% added masking.** Complete-case MLE remains close to EM despite discarding nearly two in five rows. EM uses the observed values in partial rows and achieves higher test likelihood and lower CPT KL than complete-case MLE in all three seeds. Mode imputation retains all rows, although its test likelihood falls more sharply.
- **40% added masking.** Complete-case MLE still scores close to EM with only about a third of the training rows. EM again improves both test likelihood and CPT KL over complete-case MLE in all three seeds and takes 13–14 iterations to converge.


### Overall condition score recovery accuracy

`overall_condition_score_accuracy` is the fraction of deliberately hidden test entries whose original condition-score bin matches the model's most probable bin. A value of 0.74 means 74% of those entries were recovered correctly. All methods use the same separate 20% test mask. The rows below vary the additional masking during training, and each entry is the mean across three training-mask seeds.

| Added training masking | Complete-case MLE | Mode imputation + MLE | EM |
| --- | ---: | ---: | ---: |
| 0% | 74.06% | 74.06% | 73.96% |
| 20% | 74.02% | 73.53% | 74.04% |
| 40% | 74.02% | 59.84% | 74.13% |

Missingness still affects the fitted models. From 0% to 40%, test log-likelihood falls by 0.00649 nats per row for complete-case MLE, 0.35581 for mode imputation + MLE, and 0.00292 for EM. **EM shows the smallest decline among the three methods tested.** Its gain over complete-case MLE is modest for test likelihood and individual value recovery. CPT KL measures agreement with a reference estimate fitted before additional masking. The results describe random masking of these two columns under the expert DAG. Different columns or missingness mechanisms could change the comparison.